# 🦅 Falcon-7B-Instruct — Medical QA Fine-Tuning Pipeline

**One notebook. Run top to bottom. That's it.**

All logic lives in `src/` Python modules. This notebook just calls them in order.

| Step | What happens | Module |
|------|-------------|--------|
| 1 | Install deps, clone repo, verify GPU | — |
| 2 | Load config | `src/utils.py` |
| 3 | Load tokenizer + model (4-bit quantized) | `src/model.py` |
| 4 | Load & tokenize MedQA dataset | `src/data.py` |
| 5 | Apply LoRA, build Trainer, train | `src/train.py` |
| 6 | TensorBoard monitoring | — |
| 7 | Evaluate: Perplexity · MC Accuracy · ROUGE | `src/evaluate.py` |
| 8 | Qualitative examples | `src/evaluate.py` |

> ⚡ **Before running:** Runtime → Change runtime type → **GPU (T4)**

---
## Step 1 — Install Dependencies & Clone Repo

In [ ]:
import os

REPO_URL = "https://github.com/singularity-14/finetune-falcon-7b-instruct.git"
REPO_DIR = "finetune-falcon-7b-instruct"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    print(f"📁 Repo already cloned")

%cd {REPO_DIR}

!pip install -q -r requirements-colab.txt
print("\n✅ Dependencies installed")

---
## Step 2 — Load Config & Verify GPU

In [ ]:
from src.utils import load_config, setup_logging, set_seed, gpu_report, make_output_dirs, free_memory

logger = setup_logging()
cfg    = load_config("configs/config.yaml")
set_seed(cfg["dataset"]["seed"])
make_output_dirs(cfg)

print(f"\n  Model      : {cfg['model']['name']}")
print(f"  Dataset    : {cfg['dataset']['name']}")
print(f"  Train/Eval : {cfg['dataset']['train_samples']} / {cfg['dataset']['eval_samples']}")
print(f"  LoRA rank  : {cfg['lora']['r']}")
print(f"  Epochs     : {cfg['training']['num_train_epochs']} (max, early stop patience={cfg['early_stopping']['patience']})")
print()

gpu_stats = gpu_report()

---
## Step 3 — Load Model with 4-bit Quantization + LoRA

In [ ]:
from src.model import load_tokenizer, build_quant_config, load_base_model, apply_lora

# Tokenizer
tokenizer = load_tokenizer(cfg["model"]["name"], cfg["model"]["trust_remote_code"])

# Base model (4-bit NF4)
quant_config = build_quant_config(cfg)
model = load_base_model(
    cfg["model"]["name"],
    quant_config,
    tokenizer,
    cfg["model"]["trust_remote_code"],
    cfg["model"]["use_cache"],
)

# LoRA adapter
model = apply_lora(model, cfg)

---
## Step 4 — Load & Tokenize MedQA Dataset

In [ ]:
from src.data import load_and_split

train_dataset, eval_dataset, data_collator = load_and_split(cfg, tokenizer)

---
## Step 5 — Train

In [ ]:
from src.train import build_trainer, run_training, save_artifacts

# Build Trainer
trainer = build_trainer(model, cfg, train_dataset, eval_dataset, data_collator)

# Train
train_metrics = run_training(trainer, cfg)

# Save model + adapter + eval metrics + config snapshot
final_eval = save_artifacts(trainer, model, tokenizer, cfg)

---
## Step 6 — TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {cfg['training']['logging_dir']}

---
## Step 7 — Evaluate

In [ ]:
from datasets import load_dataset
from src.evaluate import (
    compute_perplexity,
    compute_mc_accuracy,
    compute_rouge,
    save_and_print_summary,
)

DEVICE = "cuda"

# Load fresh test split (same seed ensures no data leakage)
raw = load_dataset(cfg["dataset"]["name"], split=cfg["dataset"]["split"])
test_split = raw.train_test_split(test_size=cfg["dataset"]["test_size"], seed=cfg["dataset"]["seed"])
N = cfg["evaluation"]["num_samples"]
eval_raw = test_split["test"].select(range(N))

print(f"Evaluating on {N} held-out examples...\n")

# ── 7a. Perplexity ────────────────────────────────────────────────────────────
perplexity = compute_perplexity(model, tokenizer, eval_raw, cfg, DEVICE)

# ── 7b. Multiple-Choice Accuracy ──────────────────────────────────────────────
mc_results = compute_mc_accuracy(model, tokenizer, eval_raw, cfg, DEVICE)

# ── 7c. ROUGE (on 20 samples — generation is slow) ───────────────────────────
rouge_eval = eval_raw.select(range(min(20, N)))
rouge_results = compute_rouge(model, tokenizer, rouge_eval, cfg, DEVICE)

# ── Summary ───────────────────────────────────────────────────────────────────
summary = save_and_print_summary(perplexity, mc_results, rouge_results, cfg)

---
## Step 8 — Qualitative Examples

In [ ]:
# Side-by-side: true answer vs. model-generated answer
examples = rouge_results["examples"]

print("🔍 QUALITATIVE EXAMPLES\n")
print("═" * 70)

for i, ex in enumerate(examples[:5]):
    print(f"\n[{i+1}] Question  : {ex['question'][:180]}...")
    print(f"     True Ans  : {ex['true']}")
    print(f"     Generated : {ex['generated'][:200]}")
    print(f"     ROUGE-1   : {ex['rouge1']:.3f}  |  ROUGE-L : {ex['rougeL']:.3f}")
    print("─" * 70)

---
## Step 9 — Plot Training Curves

In [ ]:
import json, math
import matplotlib.pyplot as plt
from pathlib import Path

results_dir = Path(cfg["training"]["output_dir"])
state_files = sorted(results_dir.glob("checkpoint-*/trainer_state.json"))

if state_files:
    with open(state_files[-1]) as f:
        state = json.load(f)

    history      = state.get("log_history", [])
    train_steps  = [h["step"] for h in history if "loss" in h and "eval_loss" not in h]
    train_losses = [h["loss"] for h in history if "loss" in h and "eval_loss" not in h]
    eval_epochs  = [h["epoch"]     for h in history if "eval_loss" in h]
    eval_losses  = [h["eval_loss"] for h in history if "eval_loss" in h]
    eval_ppls    = [math.exp(l)    for l in eval_losses]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Falcon-7B Fine-Tuning — Training Curves", fontsize=14, fontweight="bold")

    # Loss
    axes[0].plot(train_steps, train_losses, label="Train Loss", color="#2196F3", alpha=0.8)
    if eval_epochs:
        spe        = max(train_steps) / max(eval_epochs)
        eval_steps = [e * spe for e in eval_epochs]
        axes[0].plot(eval_steps, eval_losses, label="Eval Loss", color="#F44336", marker="o", lw=2)
    axes[0].set(xlabel="Step", ylabel="Loss", title="Training & Eval Loss")
    axes[0].legend(); axes[0].grid(alpha=0.3)

    # Perplexity
    if eval_ppls:
        axes[1].plot(eval_epochs, eval_ppls, color="#4CAF50", marker="o", lw=2)
        axes[1].set(xlabel="Epoch", ylabel="Perplexity", title="Eval Perplexity per Epoch")
        axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig("outputs/training_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("💾 Plot saved → outputs/training_curves.png")
else:
    print("⚠️  trainer_state.json not found — run training first")

---
## Step 10 — Free Memory & Download Artifacts

In [ ]:
from src.utils import free_memory
import shutil

# Free GPU
free_memory()

# Zip LoRA adapter for download
shutil.make_archive("peft_adapter", "zip", "outputs", "peft_adapter")
print("📦 peft_adapter.zip ready for download")

# Uncomment to trigger Colab download dialog:
# from google.colab import files
# files.download("peft_adapter.zip")
# files.download("outputs/evaluation_summary.json")
# files.download("outputs/training_curves.png")